# AI-Powered Music Recommendation System

This notebook demonstrates the recommendation engine behind the ChordCloud
music recommender project:

1. **Collaborative filtering** — trained on a public Last.fm user-artist
   listening-history dataset ([source](https://github.com/geeky-bit/Collaborative-filtering-for-Music-recommendation)),
   using user-based and item-based cosine similarity.
2. **Content-based filtering** — using real Spotify audio features
   (`danceability`, `energy`, `tempo`, `valence`, `acousticness`, etc.)
   from a public precomputed dataset of ~232k Spotify tracks.
3. **A hybrid recommender** blending both signals.
4. **Offline evaluation** — leave-one-out precision@k / recall@k / hit-rate@k,
   used here as a real, reproducible substitute for an unverified
   "satisfaction rate" claim.

The recommender logic lives in `music_recommender.py`, importable and
unit-testable on its own, independent of the Streamlit app.

> **A note on data sources.** Spotify deprecated the live `audio_features` /
> `audio_analysis` Web API endpoints for new third-party apps on
> November 27, 2024 — new apps get a 403 with no official replacement.
> Because of that, the content-based piece here runs on a public,
> precomputed Spotify audio-features dataset rather than a live API call.
> `ContentBasedRecommender.get_live_audio_features()` is still available
> in the module if your app happens to have grandfathered access.

## 1. Load the public Last.fm listening dataset

In [1]:
import pandas as pd
import numpy as np

from music_recommender import (
    CollaborativeFilteringRecommender,
    ContentBasedRecommender,
    HybridRecommender,
    evaluate_leave_one_out,
)

# Public dataset: binary user-artist listening matrix (1257 users x 285 artists)
# https://github.com/geeky-bit/Collaborative-filtering-for-Music-recommendation
listening_matrix = pd.read_csv("data/lastfm-matrix-germany.csv").set_index("user")
print(f"Users: {listening_matrix.shape[0]}, Artists: {listening_matrix.shape[1]}")
listening_matrix.head()

Users: 1257, Artists: 285


,a perfect circle,abba,ac/dc,adam green,aerosmith,afi,air,alanis morissette,alexisonfire,alicia keys,...,timbaland,tom waits,tool,tori amos,travis,trivium,u2,underoath,volbeat,yann tiersen
user,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
51,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
62,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. Collaborative filtering

In [2]:
cf = CollaborativeFilteringRecommender(listening_matrix)

sample_user = listening_matrix.index[0]
print(f"User-based recommendations for user {sample_user}:")
print(cf.recommend_for_user(sample_user, top_k=10))

print("\nArtists similar to 'radiohead' (item-based):")
print(cf.recommend_similar_artists("radiohead", top_k=10))

User-based recommendations for user 1:


['die toten hosen', 'subway to sally', 'queen', 'pearl jam', 'ac/dc', 'flogging molly', 'the beatles', 'in extremo', 'coldplay', 'mando diao']

Artists similar to 'radiohead' (item-based):
['mogwai', 'coldplay', 'the decemberists', 'the smashing pumpkins', 'cat power', 'the notwist', 'muse', 'portishead', 'bloc party', 'beck']


## 3. Offline evaluation (leave-one-out)

For each sampled user, one artist they actually listen to is hidden, and we
check whether the model surfaces it back in its top-k recommendations built
from the rest of their history. This gives a real, reproducible quality
metric instead of an invented "satisfaction rate".

In [3]:
results = evaluate_leave_one_out(listening_matrix, n_neighbors=10, top_k=10, n_users=300)
results

{'n_users_evaluated': 272,
 'hit_rate@k': 0.2536764705882353,
 'precision@k': 0.02536764705882353,
 'recall@k': 0.2536764705882353}

**How to report this on a resume:** cite the metric you actually computed, e.g.

> "Achieved a 25% hit-rate@10 in leave-one-out offline evaluation on a
> 1,257-user Last.fm listening-history dataset"

rather than an unverified percentage. Re-run the cell above whenever the
model or dataset changes to keep the number accurate.

## 4. Content-based filtering (Spotify audio features)

Loads a public, precomputed dataset of ~232k Spotify tracks with real
audio features (danceability, energy, tempo, valence, acousticness,
instrumentalness, speechiness, liveness) — the same attributes the live
`audio_features` endpoint used to return before its 2024 deprecation.

In [4]:
cb = ContentBasedRecommender.from_csv("data/SpotifyFeatures.csv")
print(f"Loaded {cb.features.shape[0]:,} unique tracks, {cb.features.shape[1]} audio features")

seed_id = cb.find_track_id("Bohemian Rhapsody", "Queen")
print("\nSeed track:", cb.metadata.loc[seed_id].to_dict())

similar_tracks = cb.recommend_similar_tracks(seed_id, top_k=10)
cb.metadata.loc[similar_tracks.index][["track_name", "artist_name", "genre"]].assign(
    similarity=similar_tracks.values
)

Loaded 176,774 unique tracks, 8 audio features

Seed track: {'track_name': 'Bohemian Rhapsody - Remastered 2011', 'artist_name': 'Queen', 'genre': 'Rock'}


,track_name,artist_name,genre,similarity
track_id,,,,
2rpoLjEFaFNFzHvFDf7Yd3,Forever My Friend,Ray LaMontagne,Folk,0.997012
6AWe04n4zsYkGKb3Pf7LMk,Bohemian Rhapsody - 2011 Remaster,Queen,Rock,0.996654
3lNOOfp1wgQUx06Syciz7s,Ichiban Kireina Watashiwo,Mika Nakashima,Anime,0.995708
6bMEJFQQ042ESPWSoeVGLa,Be as One,w-inds.,R&B,0.995617
47cBKOTLfTt7EOavt7ZW5V,She Talks to Angels,The Black Crowes,Blues,0.995475
2g5O5gEmjfgTDdupx7VMcp,Perfume a Tus Pies,En Espíritu Y En Verdad,Rock,0.995449
38Q0esC1cBceXOMigtCHSD,The Shortchange,Thomston,Indie,0.995419
5H3Ix0rcF6i7anJEyOCwFb,Miraiyosouzu II,DREAMS COME TRUE,Anime,0.995314
54C4TfHY00UYEnhMaYgfqZ,The Real You,Three Days Grace,Alternative,0.995261


## 5. Hybrid recommender

In [5]:
hybrid = HybridRecommender(cf, cb)

seed_id = cb.find_track_id("Creep", "Radiohead")
print("Seed track:", cb.metadata.loc[seed_id].to_dict())

hybrid_recs = hybrid.recommend(
    user_id=sample_user,
    seed_track_id=seed_id,
    cf_weight=0.4,
    top_k=10,
)
cb.metadata.loc[hybrid_recs.index][["track_name", "artist_name", "genre"]].assign(
    score=hybrid_recs.values
)

Seed track: {'track_name': 'Creep', 'artist_name': 'Radiohead', 'genre': 'Alternative'}


,track_name,artist_name,genre,score
6ImxYXeLDQPIv4qo7bMhSk,Company,Drake,Hip-Hop,0.599632
2jRRs8X0ripfTWR5ftPXSV,I'm Gucci,LightSkinKeisha,R&B,0.599496
3jSd2fWYccv84Coi2t6NOr,"Dear, Home",EXES,Dance,0.599398
3iRsGkm5n8xKB7AzOT5K41,Fear And Love,Morcheeba,Electronic,0.599357
6g6FWPdAMf55CpmkoE7jF1,Invéntame,Mon Laferte,Alternative,0.599336
1wAUM8F740gEZTyUiafTF3,Use Of Time,311,Alternative,0.599264
2m9YTbGNa7MOSXhWS7QS2i,Still I Rise,Thutmose,Soul,0.599156
5ndHl6BMrihdTqvT6Exhln,Major Bag Alert (feat. Migos),DJ Khaled,R&B,0.599124
1qylvO4iCIZZcqc4TqSjTZ,Zombie - Acoustic Version,The Cranberries,Pop,0.599073
1GLcGInllpE4JUTStzaCOM,My X,Rae Sremmurd,Hip-Hop,0.599058


## Summary

- `CollaborativeFilteringRecommender` — user-based & item-based CF on real
  listening-history data.
- `ContentBasedRecommender` — content-based filtering on real Spotify
  audio features (danceability, energy, tempo, ...), loaded from a public
  precomputed dataset since Spotify deprecated live access to this data
  for new apps in 2024. A `get_live_audio_features` path is included for
  anyone with grandfathered API access.
- `HybridRecommender` — blends both.
- `evaluate_leave_one_out` — reproducible offline quality metric.

All logic lives in `music_recommender.py` and is decoupled from the
Streamlit app so it can be developed, tested, and cited independently.